In [ ]:
from collections import defaultdict

import numpy as np

# Taller: Mejorando nuestro modelo de lenguaje

Partimos del modelo de lenguaje basado en n-gramas que construimos en
clase. En este taller vamos a estudiar algunos problemas que aparecen
cuando utilizamos este modelo para generar texto.

Trabajaremos con:

1. Smoothing
2. Backoff
3. Temperatura
4. Repetition penalty
5. Comparación de enfoques

## 0. Corpus

Usa un corpus público de texto abierto o el corpus de ejemplo de abajo.
El corpus debe tener suficiente texto para experimentar, ser coherente
y tener una licencia apropiada para uso educativo.

In [ ]:
corpus = """
el gato come pescado y el perro come carne
el gato duerme en la casa y el perro duerme en el jardin
el gato juega con la pelota y el perro juega con la pelota
el gato negro duerme en el sofa y el gato blanco come pescado
el perro grande corre en el jardin y el perro pequeno come carne
la casa tiene un jardin y el jardin tiene un arbol
el nino juega en el jardin y la nina lee un libro
la nina lee el libro y el nino come una manzana
el gato come una manzana y la nina come una manzana
""".strip()

## Modelo de n-gramas

Estas funciones ya están implementadas. Son las únicas herramientas
disponibles para el taller.

In [ ]:
def compute_ngrams(context_size: int, corpus: str) -> dict:
    model = defaultdict(lambda: defaultdict(int))
    tokens = corpus.split(" ")
    for i in range(len(tokens) - context_size):
        context = " ".join(tokens[i : i + context_size])
        model[context][tokens[i + context_size]] += 1

    probs = defaultdict(lambda: defaultdict(float))
    for context in model:
        total_count = sum(model[context].values())
        for token in model[context]:
            probs[context][token] = model[context][token] / total_count
    return probs

In [ ]:
def generate_text(
    context: str,
    model: dict[str, dict[str, float]],
    context_size: int,
    n_words: int,
) -> str:
    generated_tokens = context.split(" ")
    for _ in range(n_words):
        probs = model[context]
        next_word = str(np.random.choice(list(probs.keys()), p=list(probs.values())))
        generated_tokens.append(next_word)
        context = " ".join(generated_tokens[-context_size:])
    return " ".join(generated_tokens)

## 1. Smoothing

Con el modelo actual, si una palabra nunca apareció después de un
contexto, su probabilidad es cero y esa combinación queda descartada.

### Ejercicio

Implementa `generate_text_with_smoothing` para usar Add-k smoothing:

$$
P_k(w \mid c)
=
\frac{C(c,w)+k}
     {C(c)+kV}
$$

Prueba con:

$$
k \in \{1,\;0.1,\;0.01\}
$$

Responde:

- ¿Qué ocurre con las palabras que tenían probabilidad 0?
- ¿Qué ocurre con las palabras frecuentes?
- ¿Qué sucede con la suma de las probabilidades?

In [ ]:
def generate_text_with_smoothing(
    context: str,
    corpus: str,
    context_size: int,
    n_words: int,
    k: float,
) -> str:
    pass

## 2. Backoff

¿Qué hacemos si el contexto completo nunca apareció?

Podemos reducir progresivamente el contexto:

$$
(w_{i-2},w_{i-1}) \rightarrow (w_{i-1}) \rightarrow ()
$$

### Ejercicio

Implementa `generate_text_with_backoff` para que, cuando no encuentre
el contexto completo, utilice un contexto más pequeño.

Responde:

- ¿Qué información perdemos al reducir el contexto?
- ¿Por qué queremos utilizar el contexto más largo disponible?
- ¿Cómo se relaciona backoff con smoothing?

In [ ]:
def generate_text_with_backoff(
    context: str,
    corpus: str,
    context_size: int,
    n_words: int,
) -> str:
    pass

## 3. Temperatura

La temperatura modifica la distribución antes del sampling:

- `temperature < 1`: distribución más concentrada.
- `temperature = 1`: distribución original.
- `temperature > 1`: distribución más uniforme.

### Ejercicio

Implementa `generate_text_with_temperature`. Usa la distribución del
modelo, modifícala con la temperatura, normaliza y luego genera.

Responde:

- ¿Qué pasa con textos generados con temperaturas muy bajas?
- ¿Qué pasa con temperaturas muy altas?

In [ ]:
def generate_text_with_temperature(
    context: str,
    model: dict[str, dict[str, float]],
    context_size: int,
    n_words: int,
    temperature: float,
) -> str:
    pass

## 4. Repetition penalty

Podemos encontrar repeticiones como:

    el gato come la comida y el gato come la comida...

Una idea sencilla es penalizar las palabras que aparecieron
recientemente:

$$
P'(w \mid c)
=
\frac{P(w \mid c)}{\lambda}
$$

Después debemos volver a normalizar.

### Ejercicio

Implementa `generate_text_with_repetition_penalty`. Usa las últimas
`window_size` palabras como historial.

Prueba con:

$$
\lambda \in \{1.0,\;1.2,\;1.5,\;2.0\}
$$

Responde:

- ¿Qué ocurre cuando $\lambda = 1$?
- ¿Qué ocurre cuando $\lambda$ es demasiado grande?
- ¿Qué tamaño de ventana funciona mejor?

In [ ]:
def generate_text_with_repetition_penalty(
    context: str,
    model: dict[str, dict[str, float]],
    context_size: int,
    n_words: int,
    penalty: float,
    window_size: int,
) -> str:
    pass

## 5. Comparación final

Genera varias frases usando:

1. El modelo original.
2. El modelo con smoothing.
3. El modelo con backoff.
4. El modelo con temperatura.
5. El modelo con repetition penalty.
6. El modelo con varias técnicas combinadas.

Mantén constantes las demás condiciones de generación para que la
comparación sea lo más justa posible.

Observa:

- coherencia;
- variedad;
- repeticiones;
- palabras inesperadas.

## 6. Reto final

Elige una configuración de:

- tamaño de n-grama;
- smoothing;
- backoff;
- temperatura;
- repetition penalty.

que produzca los mejores resultados según tu criterio.

Genera varias frases y explica por qué elegiste esa configuración.

## 7. Entrega

Envie su solucion en [este enlace](https://forms.gle/WRDi6SRN9Dtdu97H9)